### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import re

#train_path = kagglehub.competition_download('drawing-with-llms', 'train.csv')
df = pd.read_csv('./drawing-with-llms/svg_score_train1.csv')
df=df[df['score'] > 0.5]
print(df.shape)
df.head(2)

(1062, 4)


,description,svg_code,response,score
0,"'Golden wheat fields under a setting sun',","<svg viewBox=""0 0 200 200"" width=""200"" height=...","```svg\n<svg viewBox=""0 0 200 200"" width=""200""...",0.994973
3,"'Snowy mountains under a clear blue sky',","<svg viewBox=""0 0 200 100"" width=""200"" height=...","```svg\n<svg viewBox=""0 0 200 100"" width=""200""...",0.978844


In [2]:
import openai
import pandas as pd
import time
import os
from dotenv import load_dotenv
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")

In [3]:
from openai import OpenAI
client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hello are you"},
    ],
    stream=False
)

print(response.choices[0].message.content)

Hello! Yes, I'm here and ready to help. How can I assist you today? 😊


In [4]:
# Function to get summary from OpenAI API
def get_svg_code(previous_svg,text):
    instruction = f"""
            Generate SVG code to visually represent the following text description, while respecting the given constraints.
            <constraints>
            * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
            * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
            </constraints>

            Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints. 
            Focus on a clear and concise representation of the input description within the given limitations. 
            Always give the complete SVG code with nothing omitted. Never use an ellipsis.

            The code is scored based on similarity to the description, Visual question anwering and aesthetic components. here is the previously generated code 
            that lacks in aesthetic part here: {previous_svg}.

            Please generate new code considering similarity, visual question answering and aesthetic compoenents.

            input description: {text}
            """

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "Generate SVG code as per instruction"},
                {"role": "user", "content": instruction}
            ],
            temperature=0.6,
            max_tokens=2560
        )
        return response.choices[0].message.content.strip()
    
    except Exception as e:
        print(f"Error: {e}")
        return None

In [5]:
from tqdm import tqdm
tqdm.pandas()
df["response_2"] = df.progress_apply(lambda row: get_svg_code(row["svg_code"], row["description"]), axis=1)

100%|████████████████████████████████████| 1062/1062 [12:19:08<00:00, 41.76s/it]


In [7]:
df.to_csv('response_2_1062.csv',index=False)